In [19]:
import os
import re
import random
import time
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from bert_score import score as bertscore_score

In [20]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [21]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(4)

In [22]:
DATA_DIR = 'Data_to_Students'
OUT_DIR = '.'
# Force CPU for stability (MPS can hang on some operations)
DEVICE = torch.device('cpu')
print('Using device: CPU (for stability)')

Using device: CPU (for stability)


In [23]:
def tokenize(text):
    text = text.strip().lower().replace('“', '"').replace('”', '"').replace('’', "'")
    return re.findall(r"[\w]+|[^\w\s]", text)

In [24]:
def load_parallel(path_src, path_tgt):
    src_df = pd.read_csv(path_src)
    tgt_df = pd.read_csv(path_tgt)
    src_df = src_df.rename(columns={src_df.columns[0]: 'id', src_df.columns[1]: 'src'})
    tgt_df = tgt_df.rename(columns={tgt_df.columns[0]: 'id', tgt_df.columns[1]: 'tgt'})
    merged = src_df.merge(tgt_df, on='id')
    return merged[['id', 'src', 'tgt']].copy()

In [25]:
class Vocab:
    def __init__(self, sentences, max_vocab=6000):
        counts = {}
        for sent in sentences:
            for tok in sent:
                counts[tok] = counts.get(tok, 0) + 1
        special = ['<pad>', '<sos>', '<eos>', '<unk>']
        sorted_tokens = sorted(counts.items(), key=lambda x: (-x[1], x[0]))
        vocab_tokens = special + [tok for tok, _ in sorted_tokens[:max_vocab - len(special)]]
        self.word2idx = {tok: i for i, tok in enumerate(vocab_tokens)}
        self.idx2word = {i: tok for tok, i in self.word2idx.items()}
        self.pad_idx = self.word2idx['<pad>']
        self.sos_idx = self.word2idx['<sos>']
        self.eos_idx = self.word2idx['<eos>']
        self.unk_idx = self.word2idx['<unk>']

    def encode(self, tokens):
        return [self.word2idx.get(tok, self.unk_idx) for tok in tokens]

    def decode(self, ids):
        return [self.idx2word.get(i, '<unk>') for i in ids]

In [26]:
class NMTDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences):
        self.src_sentences = src_sentences
        self.tgt_sentences = tgt_sentences

    def __len__(self):
        return len(self.src_sentences)

    def __getitem__(self, idx):
        return self.src_sentences[idx], self.tgt_sentences[idx]

In [27]:
def collate_fn(batch, src_pad_idx, tgt_pad_idx):
    src_seqs, tgt_seqs = zip(*batch)
    src_lengths = torch.tensor([len(s) for s in src_seqs], dtype=torch.long)
    src_padded = torch.nn.utils.rnn.pad_sequence([torch.tensor(s, dtype=torch.long) for s in src_seqs], batch_first=True, padding_value=src_pad_idx)
    tgt_padded = torch.nn.utils.rnn.pad_sequence([torch.tensor(t, dtype=torch.long) for t in tgt_seqs], batch_first=True, padding_value=tgt_pad_idx)
    return src_padded, src_lengths, tgt_padded

In [28]:
class Attention(nn.Module):
    def __init__(self, enc_hidden_dim, dec_hidden_dim):
        super().__init__()
        self.attn = nn.Linear(enc_hidden_dim * 2 + dec_hidden_dim, dec_hidden_dim)
        self.v = nn.Linear(dec_hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs, mask=None):
        batch_size, src_len, _ = encoder_outputs.size()
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn_weights = F.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        return context, attn_weights

In [29]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, enc_hidden_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim, padding_idx=0)
        self.rnn = nn.LSTM(emb_dim, enc_hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))
        packed = nn.utils.rnn.pack_padded_sequence(embedded, src_lengths.cpu(), batch_first=True, enforce_sorted=False)
        outputs, (hidden, cell) = self.rnn(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs, batch_first=True)
        return outputs, hidden, cell

In [30]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hidden_dim, dec_hidden_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim, padding_idx=0)
        self.attention = Attention(enc_hidden_dim, dec_hidden_dim)
        self.rnn = nn.LSTM(emb_dim + enc_hidden_dim * 2, dec_hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(dec_hidden_dim + enc_hidden_dim * 2 + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_step, hidden, cell, encoder_outputs, mask=None):
        embedded = self.dropout(self.embedding(input_step))
        context, attn_weights = self.attention(hidden[-1], encoder_outputs, mask)
        context = context.unsqueeze(1)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        output = output.squeeze(1)
        context = context.squeeze(1)
        embedded = embedded.squeeze(1)
        pred = self.fc_out(torch.cat((output, context, embedded), dim=1))
        return pred, hidden, cell, attn_weights

In [31]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, tgt_vocab):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.tgt_vocab = tgt_vocab
        self.enc_hidden_dim = encoder.rnn.hidden_size
        self.fc_hidden = nn.Linear(self.enc_hidden_dim * 2, decoder.rnn.hidden_size)
        self.fc_cell = nn.Linear(self.enc_hidden_dim * 2, decoder.rnn.hidden_size)

    def create_mask(self, src):
        return (src != self.tgt_vocab.pad_idx).to(src.device)

    def forward(self, src, src_lengths, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.size(0)
        max_len = tgt.size(1)
        trg_vocab_size = len(self.tgt_vocab.word2idx)
        outputs = torch.zeros(batch_size, max_len, trg_vocab_size, device=src.device)

        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
        cell = torch.tanh(self.fc_cell(torch.cat((cell[-2], cell[-1]), dim=1))).unsqueeze(0)
        input_step = torch.full((batch_size, 1), self.tgt_vocab.sos_idx, dtype=torch.long, device=src.device)
        mask = self.create_mask(src)

        for t in range(max_len):
            output, hidden, cell, _ = self.decoder(input_step, hidden, cell, encoder_outputs, mask)
            outputs[:, t, :] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            input_step = tgt[:, t].unsqueeze(1) if teacher_force else top1

        return outputs

    def greedy_decode(self, src, src_lengths, max_len=40):
        batch_size = src.size(0)
        encoder_outputs, hidden, cell = self.encoder(src, src_lengths)
        hidden = torch.tanh(self.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
        cell = torch.tanh(self.fc_cell(torch.cat((cell[-2], cell[-1]), dim=1))).unsqueeze(0)
        input_step = torch.full((batch_size, 1), self.tgt_vocab.sos_idx, dtype=torch.long, device=src.device)
        mask = self.create_mask(src)
        decoded_ids = [[] for _ in range(batch_size)]
        for _ in range(max_len):
            output, hidden, cell, _ = self.decoder(input_step, hidden, cell, encoder_outputs, mask)
            top1 = output.argmax(1).unsqueeze(1)
            input_step = top1
            for i, idx in enumerate(top1.squeeze(1).tolist()):
                decoded_ids[i].append(idx)
        return decoded_ids

In [32]:
class BeamSearchDecoder:
    def __init__(self, model, beam_width=3, max_len=40):
        self.model = model
        self.beam_width = beam_width
        self.max_len = max_len

    def decode(self, src, src_lengths):
        self.model.eval()
        with torch.no_grad():
            encoder_outputs, hidden, cell = self.model.encoder(src, src_lengths)
            hidden = torch.tanh(self.model.fc_hidden(torch.cat((hidden[-2], hidden[-1]), dim=1))).unsqueeze(0)
            cell = torch.tanh(self.model.fc_cell(torch.cat((cell[-2], cell[-1]), dim=1))).unsqueeze(0)
            mask = self.model.create_mask(src)
            batch_size = src.size(0)
            results = []
            for i in range(batch_size):
                beams = [([self.model.tgt_vocab.sos_idx], 0.0, hidden[:, i:i+1, :].contiguous(), cell[:, i:i+1, :].contiguous())]
                completed = []
                for _ in range(self.max_len):
                    new_beams = []
                    for seq, score, h, c in beams:
                        if seq[-1] == self.model.tgt_vocab.eos_idx:
                            completed.append((seq, score))
                            continue
                        input_step = torch.tensor([[seq[-1]]], device=src.device)
                        output, nh, nc, _ = self.model.decoder(input_step, h, c, encoder_outputs[i:i+1], mask[i:i+1])
                        log_probs = F.log_softmax(output, dim=1).squeeze(0)
                        topk = log_probs.topk(self.beam_width)
                        for prob, idx in zip(topk.values.tolist(), topk.indices.tolist()):
                            new_beams.append((seq + [idx], score + prob, nh, nc))
                    beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:self.beam_width]
                    if not beams:
                        break
                if completed:
                    seq, _ = max(completed, key=lambda x: x[1])
                else:
                    seq, _ = max(beams, key=lambda x: x[1])
                results.append(seq[1:])
            return results

In [33]:
def train_model(train_pairs, src_vocab, tgt_vocab, batch_size=64, epochs=25):
    src_train = [src_vocab.encode(tokenize(s)) for s, _ in train_pairs]
    # Fix: properly encode target with sos and eos tokens
    tgt_train = [[tgt_vocab.sos_idx] + tgt_vocab.encode(tokenize(t)) + [tgt_vocab.eos_idx] for _, t in train_pairs]
    dataset = NMTDataset(src_train, tgt_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx, tgt_vocab.pad_idx), num_workers=0)

    encoder = Encoder(len(src_vocab.word2idx), emb_dim=256, enc_hidden_dim=256, dropout=0.3)
    decoder = Decoder(len(tgt_vocab.word2idx), emb_dim=256, enc_hidden_dim=256, dec_hidden_dim=256, dropout=0.3)
    model = Seq2Seq(encoder, decoder, tgt_vocab).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab.pad_idx)

    print(f"Starting training on {DEVICE}...")
    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for i, (src, src_lengths, tgt) in enumerate(loader):
            src, src_lengths, tgt = src.to(DEVICE), src_lengths.to(DEVICE), tgt.to(DEVICE)
            optimizer.zero_grad()
            # Reduce teacher forcing gradually
            teacher_forcing_ratio = max(0.5, 1.0 - epoch * 0.1)
            outputs = model(src, src_lengths, tgt, teacher_forcing_ratio=teacher_forcing_ratio)
            loss = criterion(outputs.view(-1, outputs.size(-1)), tgt.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            
            # Print progress every 20 batches
            if i % 20 == 0:
                print(f"  epoch {epoch+1}/{epochs} batch {i}/{len(loader)} loss {loss.item():.4f}")
        
        print(f'epoch {epoch+1}/{epochs} avg_loss {total_loss / (i+1):.4f}')
    return model

In [34]:
def decode_sentence(model, sentence, src_vocab, tgt_vocab, max_len=40):
    model.eval()
    ids = src_vocab.encode(tokenize(sentence))
    src_tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    src_lengths = torch.tensor([len(ids)], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        decoded_ids = model.greedy_decode(src_tensor, src_lengths, max_len=max_len)[0]
    tokens = [tgt_vocab.idx2word.get(idx, '<unk>') for idx in decoded_ids if idx not in {tgt_vocab.eos_idx, tgt_vocab.pad_idx, tgt_vocab.sos_idx}]
    return ' '.join(tokens)

In [35]:
def clean_text(text):
    return ' '.join(text.replace('``', '"').replace('`', '"').split())

In [36]:
if __name__ == '__main__':
    train_df = load_parallel(os.path.join(DATA_DIR, 'train_sa_10000.csv'), os.path.join(DATA_DIR, 'train_en_10000.csv'))
    dev_df = load_parallel(os.path.join(DATA_DIR, 'dev_sa_1000.csv'), os.path.join(DATA_DIR, 'dev_en_1000.csv'))
    test_df = load_parallel(os.path.join(DATA_DIR, 'test_sa_1000.csv'), os.path.join(DATA_DIR, 'test_en_1000.csv'))

    train_src = [tokenize(s) for s in train_df['src'].tolist()]
    train_tgt = [tokenize(t) for t in train_df['tgt'].tolist()]
    dev_src = [tokenize(s) for s in dev_df['src'].tolist()]
    dev_tgt = [tokenize(t) for t in dev_df['tgt'].tolist()]

    src_vocab = Vocab(train_src + dev_src, max_vocab=6000)
    tgt_vocab = Vocab(train_tgt + dev_tgt, max_vocab=6000)

    train_pairs = list(zip(train_df['src'].tolist(), train_df['tgt'].tolist()))
    model = train_model(train_pairs, src_vocab, tgt_vocab, batch_size=64, epochs=25)

    dev_preds = [clean_text(decode_sentence(model, src, src_vocab, tgt_vocab)) for src in dev_df['src'].tolist()]
    dev_refs = [clean_text(' '.join(tokenize(t))) for t in dev_df['tgt'].tolist()]

    smoothie = SmoothingFunction().method1
    bleu = corpus_bleu([[ref.split()] for ref in dev_refs], [pred.split() for pred in dev_preds], smoothing_function=smoothie)
    print('DEV BLEU', bleu)

    P, R, F1 = bertscore_score(dev_preds, dev_refs, lang='en', model_type='distilbert-base-uncased', batch_size=16, device='cpu', verbose=False)
    print('DEV BERT F1', float(F1.mean()))

    start_time = time.time()
    test_preds = [clean_text(decode_sentence(model, src, src_vocab, tgt_vocab)) for src in test_df['src'].tolist()]
    elapsed = time.time() - start_time

    print(f'Inference time: {elapsed:.2f} seconds')
    print(f'Total model parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}')

    submission = pd.DataFrame({'Source_id': test_df['id'].tolist(), 'Sentence_en': test_preds})
    submission.to_csv(os.path.join(OUT_DIR, 'submission.csv'), index=False, encoding='utf-8')
    print('Saved submission.csv')

Starting training on cpu...
  epoch 1/25 batch 0/157 loss 8.8208
  epoch 1/25 batch 20/157 loss 5.5024
  epoch 1/25 batch 40/157 loss 5.1733
  epoch 1/25 batch 60/157 loss 4.9253
  epoch 1/25 batch 80/157 loss 4.9566
  epoch 1/25 batch 100/157 loss 4.7754
  epoch 1/25 batch 120/157 loss 4.6853
  epoch 1/25 batch 140/157 loss 4.7515
epoch 1/25 avg_loss 5.1468
  epoch 2/25 batch 0/157 loss 4.4719
  epoch 2/25 batch 20/157 loss 4.3310
  epoch 2/25 batch 40/157 loss 4.2028
  epoch 2/25 batch 60/157 loss 4.3038
  epoch 2/25 batch 80/157 loss 4.4974
  epoch 2/25 batch 100/157 loss 4.1811
  epoch 2/25 batch 120/157 loss 4.1902
  epoch 2/25 batch 140/157 loss 4.2019
epoch 2/25 avg_loss 4.3136
  epoch 3/25 batch 0/157 loss 4.4409
  epoch 3/25 batch 20/157 loss 4.0512
  epoch 3/25 batch 40/157 loss 4.2483
  epoch 3/25 batch 60/157 loss 3.8606
  epoch 3/25 batch 80/157 loss 3.7527
  epoch 3/25 batch 100/157 loss 4.0225
  epoch 3/25 batch 120/157 loss 4.1455
  epoch 3/25 batch 140/157 loss 3.7514


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10841.08it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DEV BERT F1 0.769096314907074
Inference time: 13.29 seconds
Total model parameters: 11785072
Saved submission.csv
